# Sedimentation–Crystallisation Model

## Geometry and ingredients

The system is a vertical capillary of cross-section $A$ and initial height $H$.  
Three regions are tracked at every instant:

| Region | Extent | Packing fraction |
|--------|--------|-----------------|
| Supernatant | $h_\text{clear} \to H$ | $\approx 0$ (clear) |
| Suspension | $h_\text{sed} \to h_\text{clear}$ | $\phi_\text{fluid}(t)$ |
| Sediment | $0 \to h_\text{sed}$ | $\phi_\text{sed}$ (fixed) |

Two tracked interfaces: clear-fluid height $h_\text{clear}$ and sediment height $h_\text{sed}$.

Initial conditions: $h_\text{clear}(0)=H$, $h_\text{sed}(0)=0$, $\phi_\text{fluid}(0)=\phi_0$.





## Assumptions

- **Hard-sphere phase diagram**: freezing $\phi_F = 0.494$, melting $\phi_M = 0.545$.
- **Hindered settling** of background spheres follows the Hayakawa–Ichiki expression.
- **Nucleation rate** is taken from Auer & Frenkel (2001) simulation data (monodisperse, Fig. 2); extrapolated linearly in $\log_{10} J$ vs $\phi$.
- **Nucleus size** depends on packing fraction via a fitted power-law diverging at $\phi_\text{div} = 0.4917$: $N_\text{nuc}(\phi) = a\,(\phi - \phi_\text{div})^{-b}$, with $a = 0.00566$, $b = 3.016$.
- **Crystal growth** proceeds by a flat-interface model: the interface advances at a constant speed $v_g$.
- **Crystal sedimentation** follows Stokes law for a sphere of radius $R$, corrected for the buoyancy contrast $\Delta\phi = \phi_M - \phi_\text{fluid}$ and hindered by the suspension via the Hayakawa–Ichiki factor.
- **Sediment** is incompressible with fixed $\phi_\text{sed}$.
- Particle number is globally conserved; $\phi_\text{fluid}$ is updated at every step from a global mass balance.



## Governing equations

### Background settling and interface motion

Single-particle terminal velocity:
$$v_0 = \mathrm{Pe}\,\frac{\sigma}{\tau_B}$$

Hayakawa–Ichiki hindered-settling factor:
$$K(\phi) = \frac{(1-\phi)^3}{1 + 2\phi + 1.492\,\phi(1-\phi)^3}$$

Suspension sedimentation velocity:
$$v_\text{sed}(\phi) = v_0\,K(\phi)$$

Clear-fluid interface drops at:
$$\frac{dh_\text{clear}}{dt} = -v_\text{sed}(\phi_\text{fluid})$$

Sediment interface rises (mass conservation at the sediment front):
$$\frac{dh_\text{sed}}{dt} = \frac{\phi_\text{fluid}}{\phi_\text{sed} - \phi_\text{fluid}}\,v_\text{sed}(\phi_\text{fluid}) + \frac{v_\text{wall}}{A}$$

where $v_\text{wall} = 2\,v_g\,(\phi_\text{fluid}-\phi_F)\,\Delta t\,(h_\text{clear}-h_\text{sed})$ accounts for wall-heterogeneous crystallisation.

### Nucleation

Homogeneous nucleation rate density (Auer & Frenkel fit):
$$\log_{10} J(\phi) = a\,\phi + b \quad [\sigma^{-3}\,\tau_B^{-1}]$$

calibrated to $\{(0.52,\,-18.5),\,(0.53,\,-13),\,(0.535,\,-8.5)\}$.

Expected number of new nuclei per step:
$$\lambda = J(\phi_\text{fluid})\,A\,(h_\text{clear}-h_\text{sed})\,\Delta t$$

Actual count drawn from $\text{Poisson}(\lambda)$; nuclei placed uniformly in $[h_\text{sed}, h_\text{clear}]$.

Nucleus size (number of particles) from a fitted power-law:
$$N_\text{nuc}(\phi) = \frac{a}{(\phi - \phi_\text{div})^{b}}, \qquad \phi_\text{div}=0.4917,\; a=0.00566,\; b=3.016$$

Nucleus volume: $V_\text{nuc} = N_\text{nuc}(\phi)\,V_\text{sphere}/\phi_M$.

### Crystal sedimentation

For a crystal of volume $V$ (radius $R = (3V/4\pi)^{1/3}$):

$$v_\text{crystal} = \frac{4\,R^2}{\sigma\,\tau_B}\,\mathrm{Pe}^{\,\Delta\phi}\,K(\phi_\text{fluid}), \qquad \Delta\phi = \phi_M - \phi_\text{fluid}$$

### Crystal growth

$$\frac{dV}{dt} = 4\pi R^2\,v_g, \qquad \text{if } \phi_\text{fluid} \ge \phi_F$$

### Mass balance (solved each step for $\phi_\text{fluid}$)

$$\phi_\text{fluid} = \frac{N_\text{total} - \phi_\text{sed}\,A\,h_\text{sed} - \phi_M\,V_\text{crystals}}{V_\text{fluid}}$$

where $V_\text{fluid} = A(h_\text{clear}-h_\text{sed}) - V_\text{crystals}$.





## Algorithm

1. **Bulk settling** — advance $h_\text{clear}$ downward by $v_\text{sed}(\phi_\text{fluid})\,\Delta t$.
2. **Sediment growth** — advance $h_\text{sed}$ upward via the mass-conservation condition plus wall-crystallisation term.
3. **Nucleation** — draw $n \sim \text{Poisson}(\lambda)$; append $n$ new crystals at random positions in the suspension region.
4. **Crystal update** (for each crystal):
   - Grow volume: $V \mathrel{+}= (dV/dt)\,\Delta t$.
   - Sediment: $x \mathrel{-}= v_\text{crystal}\,\Delta t$.
   - If $x \le h_\text{sed}$: transfer crystal to sediment, increment $h_\text{sed}$ accordingly, remove from list.
5. **Mass balance** — recompute $\phi_\text{fluid}$ from global conservation.
6. **Log** — store $t,\,h_\text{clear},\,h_\text{sed},\,\phi_\text{fluid},\,n_\text{crystals}$, etc.
7. **Repeat** until $h_\text{clear} \le h_\text{sed}$ or target steps reached.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class Crystal:
    def __init__(self, x, vol):
        self.x = x       # Vertical position (sigma)
        self.vol = vol   # Volume (sigma^3)


# --- Pure mathematical/utility functions ---

def hayakawa_ichki_expr(phi):
    """Hayakawa-Ichki expression for hindered settling."""
    return (1.0 - phi)**3 / (1.0 + 2.0*phi + 1.492*phi*(1.0 - phi)**3)

def nucleus_model(phi):
    a, b = np.array([0.00566154, 3.01579022])
    divergence = 0.4917
    return a/(phi-divergence)**b

def sedimentation_speed_richardson_zaki(phi, vol, V0_SPHERE, D_SPHERE):
    """Richardson–Zaki hindered settling velocity [sigma/s]."""
    l = (6 * vol / np.pi)**(1/3)
    v0 = V0_SPHERE * (l / D_SPHERE)**2
    return v0 * (1.0 - phi)**4.65


def sedimentation_speed_hayakawa_ichki(phi, vol, V0_SPHERE, D_SPHERE):
    """Hayakawa-Ichki hindered settling velocity [sigma/s]."""
    l = (6 * vol / np.pi)**(1/3)
    v0 = V0_SPHERE * (l / D_SPHERE)**2
    return v0 * (1.0 - phi)**3 / (1.0 + 2.0*phi + 1.492*phi*(1.0 - phi)**3)


def sedimentation_speed(phi, vol, V0_SPHERE, D_SPHERE, model='hayakawa_ichki'):
    """Hindered settling velocity with choice of model [sigma/s]."""
    if model == 'richardson_zaki':
        return sedimentation_speed_richardson_zaki(phi, vol, V0_SPHERE, D_SPHERE)
    elif model == 'hayakawa_ichki':
        return sedimentation_speed_hayakawa_ichki(phi, vol, V0_SPHERE, D_SPHERE)
    else:
        raise ValueError(f"Unknown sedimentation model: {model}")


class CrystallizationSimulation:
    """Simulate sedimentation and crystallisation in a colloidal suspension."""

    def __init__(self, Pe=0.13, phi_initial=0.53, nparticle_per_side=10000, phi_sediment=0.55):
        self.sigma = 1.0
        self.brownian_time = 1.0

        self.H_TOP = 3e4 * self.sigma
        self.PHI_FREEZE = 0.494
        self.PHI_MELT = 0.545
        self.PHI_SEDIMENT = phi_sediment
        self.D_SPHERE = self.sigma

        self.Pe = Pe
        self.V0_SPHERE = Pe * self.sigma / self.brownian_time
        self.LINEAR_GROWTH = 0.2 * self.sigma / self.brownian_time
        self.AREA = (nparticle_per_side * self.sigma)**2
        self.KN = 1e-11 / self.brownian_time / self.sigma**3

        self.N_PARTICLES_NUCLEUS = 150
        self.SPHERE_VOLUME = np.pi / 6 * self.D_SPHERE**3
        self.V_NUC = self.N_PARTICLES_NUCLEUS * self.SPHERE_VOLUME / self.PHI_MELT

        self.V_BG = self.SPHERE_VOLUME
        self.phi_initial = phi_initial
        self.history = None

    def nucleation_rate(self, phi, sim=True, mono=True):
        """Nucleation rate density [sigma^-3 tau_B^-1] from Auer & Frenkel (2001)."""
        if sim and mono:
            phis = np.array([0.52, 0.53, 0.535])
            log10J = np.array([-18.5, -13, -8.5])
            lfit = np.polyfit(phis, log10J, 1)
            return 10**(lfit[0]*phi + lfit[1])
        return 0.0

    def growth_rate(self, phi, vol=None):
        """Volumetric growth rate per crystal [sigma^3 s^-1]."""
        if phi < self.PHI_FREEZE:
            return 0.0
        if vol is None:
            return 0.0
        r = (3 * vol / (4 * np.pi))**(1/3)
        surface_area = 4 * np.pi * r**2
        return surface_area * self.LINEAR_GROWTH

    def simulate(self, dt, total_steps, sedimentation_model='hayakawa_ichki', empirical_effective_prefactor=0.1):
        """Simulate sedimentation and crystallisation."""
        h_clear = self.H_TOP
        h_sed   = 0.0
        phi_fluid = self.phi_initial

        N_TOTAL = self.phi_initial * self.AREA * self.H_TOP
        crystals = []

        self.history = {
            'time': [], 'h_clear': [], 'h_sed': [],
            'phi_fluid': [], 'n_crystals': [], 'total_N': [],
            'new_crystals': [], 'landed_crystals': [],
            'crystalline_fraction_of_suspension': []
        }

        for step in range(total_steps):
            vol_solution = self.AREA * (h_clear - h_sed)
            if vol_solution <= 0:
                print("Solution fully crystallized or sedimented. Ending simulation.")
                break

            # 1. Background Settling
            v_bulk = sedimentation_speed(phi_fluid, self.V_BG, self.V0_SPHERE, self.D_SPHERE, model=sedimentation_model)
            h_clear -= v_bulk * dt

            v_wall = 2*self.LINEAR_GROWTH*(phi_fluid-self.PHI_FREEZE) * dt*(h_clear-h_sed)
            dh_wall = v_wall/self.AREA

            dh = v_bulk * (phi_fluid / (self.PHI_SEDIMENT - phi_fluid)) * dt + dh_wall
            if h_sed+dh > h_clear:
                print("Ending simulation.")
                break
            else:
                h_sed += dh

            # 2. Nucleation
            J = self.nucleation_rate(phi_fluid)

            num_nucleus = nucleus_model(phi_fluid)
            nucleus_volume = num_nucleus * self.SPHERE_VOLUME / self.PHI_MELT

            lam = J * vol_solution * dt
            if lam > 0:
                new_nuclei = np.random.poisson(lam)
                self.history['new_crystals'].append(new_nuclei)
                for _ in range(new_nuclei):
                    crystals.append(Crystal(x=np.random.uniform(h_sed, h_clear), vol=nucleus_volume))
            else:
                self.history['new_crystals'].append(0)

            # 3. Crystal Growth and Settling
            total_vol_growth = 0.0
            n_landed = 0
            for c in crystals[:]:
                dv = self.growth_rate(phi_fluid, vol=c.vol) * dt
                c.vol += dv
                total_vol_growth += dv

                dphi = self.PHI_MELT - phi_fluid
                R = (3 * c.vol / (4 * np.pi))**(1/3)

                v_sed = 4/self.brownian_time*R**2/self.sigma*self.Pe**dphi*hayakawa_ichki_expr(phi_fluid)

                if v_sed < v_bulk:
                    print(f"Warning: sedimentation speed {v_sed:.2e} < bulk speed {v_bulk:.2e} at step {step}.")

                c.x -= v_sed * dt

                if c.x <= h_sed:
                    h_sed += (c.vol * self.PHI_MELT) / (self.AREA * self.PHI_SEDIMENT)
                    crystals.remove(c)
                    n_landed += 1

            # 4. Mass Balance
            vol_crystals = sum(c.vol for c in crystals)
            vol_fluid = self.AREA * (h_clear - h_sed) - vol_crystals
            vol_crystals += v_wall

            phi_fluid = (N_TOTAL - self.PHI_SEDIMENT * self.AREA * h_sed - vol_crystals*self.PHI_MELT) / vol_fluid

            self.history['landed_crystals'].append(n_landed)

            assert phi_fluid >= 0, f"Negative fluid packing fraction: {phi_fluid:.3f} at step {step}"

            crystalline_fraction_of_suspension = sum(c.vol for c in crystals)/vol_solution
            self.history['crystalline_fraction_of_suspension'].append(crystalline_fraction_of_suspension)

            total_N = phi_fluid * vol_fluid + h_sed*self.AREA*self.PHI_SEDIMENT + sum(c.vol for c in crystals)*self.PHI_MELT

            if step % 1000 == 0:
                print(f"Step {step} | h_clear: {h_clear:.2f} | h_sed: {h_sed:.2f} | phi_fluid: {phi_fluid:.3f} | n_crystals: {len(crystals)} | N ratio: {total_N/N_TOTAL:.2e}")

            # 5. Logging
            self.history['time'].append(step * dt)
            self.history['h_clear'].append(h_clear)
            self.history['h_sed'].append(h_sed)
            self.history['phi_fluid'].append(phi_fluid)
            self.history['n_crystals'].append(len(crystals))
            self.history['total_N'].append(total_N)

        for key in self.history:
            self.history[key] = np.array(self.history[key])

    def plot_overview(self, figsize=(12, 7), brownian_time_in_seconds=None):
        """Plot overview of simulation with four subplots."""
        if self.history is None:
            raise ValueError("No simulation history. Run simulate() first.")
        if brownian_time_in_seconds is None:
            conversion_factor = 1
            time_label = r'Time ($\tau_B$)'
        else:
            conversion_factor = brownian_time_in_seconds/self.brownian_time/60/60/24
            time_label = 'Time (days)'

        fig = plt.figure(figsize=figsize)
        gs = fig.add_gridspec(2, 2)
        ax_top = fig.add_subplot(gs[0, :])
        ax_bl = fig.add_subplot(gs[1, 0], sharex=ax_top)
        ax_br = fig.add_subplot(gs[1, 1], sharex=ax_top)

        ax_top.plot(self.history['time']*conversion_factor, self.history['h_clear'], color='steelblue', label=r'$h_\mathrm{clear}$')
        ax_top.plot(self.history['time']*conversion_factor, self.history['h_sed'], color='darkorange', label=r'$h_\mathrm{sed}$')
        ax_top.set_ylabel(r'Height ($\sigma$)')
        ax_top.set_xlabel(time_label)
        ax_top.legend()

        ax_bl.scatter(self.history['time']*conversion_factor, self.history['n_crystals'], color='seagreen', s=2)
        ax_bl.set_xlabel(time_label)
        ax_bl.set_ylabel('Count')
        ax_bl.set_title('Number of suspended crystals')

        ax_br.plot(self.history['time']*conversion_factor, self.history['phi_fluid'], color='mediumpurple')
        ax_br.set_xlabel(time_label)
        ax_br.set_ylabel(r'$\phi$')
        ax_br.set_title('Fluid packing fraction in solution region')

        plt.tight_layout()
        plt.show()

    def plot_particle_conservation(self):
        if self.history is None:
            raise ValueError("No simulation history. Run simulate() first.")
        N_initial = self.phi_initial * self.AREA * self.H_TOP
        plt.figure()
        plt.plot(self.history['time'], self.history['total_N'] / N_initial, color='crimson')
        plt.xlabel(r'Time ($\tau_B$)')
        plt.ylabel('Total N (normalized)')
        plt.title('Total number of particles (fluid + crystals + sediment)')
        plt.show()

    def plot_nucleation(self):
        if self.history is None:
            raise ValueError("No simulation history. Run simulate() first.")
        plt.figure()
        plt.plot(self.history['time'], self.history['new_crystals'], '-', color='teal')
        plt.xlabel(r'Time ($\tau_B$)')
        plt.ylabel('Count')
        plt.title('New nuclei per step')
        plt.show()

    def plot_crystalline_fraction(self):
        if self.history is None:
            raise ValueError("No simulation history. Run simulate() first.")
        plt.figure()
        plt.plot(self.history['time'], self.history['crystalline_fraction_of_suspension'], color='purple')
        plt.xlabel(r'Time ($\tau_B$)')
        plt.ylabel('Crystalline fraction')
        plt.title('Crystalline fraction of the suspension')
        plt.show()

In [ ]:
PHI_INITIAL = 0.53
sim = CrystallizationSimulation(Pe=0.13, phi_initial=PHI_INITIAL, nparticle_per_side=1000, phi_sediment=0.74)
sim.simulate(dt=1.0, total_steps=2_000_000)

In [ ]:
skip = 50_000
fit = np.polyfit(sim.history["time"][skip:], sim.history['h_sed'][skip:], 1)
intercept = fit[1]
dphi = sim.PHI_MELT - sim.PHI_FREEZE
pa_phi_initial = intercept/sim.H_TOP * dphi + sim.PHI_FREEZE

print(f"Linear-fit intercept: {intercept:.2f}")
print(f"Lever-rule estimate of phi_initial: {pa_phi_initial:.3f}")
print(f"Input phi_initial:                  {sim.phi_initial:.3f}")

sim.plot_overview(brownian_time_in_seconds=1.3)

plt.figure()
plt.plot(sim.history["time"], sim.history['h_sed'], label=r'$h_\mathrm{sed}$')
plt.plot(sim.history["time"], fit[0]*sim.history["time"] + fit[1], '--', label='linear fit')
plt.xlabel(r'Time ($\tau_B$)')
plt.ylabel(r'$h_\mathrm{sed}$ ($\sigma$)')
plt.legend()
plt.show()